# Ice cream sales model, version 2

Version 1 predicted sales from **temperature** only. This version adds two more things that plausibly affect ice cream sales:

- **rain_chance**: the forecast chance of rain that day (0 to 100 %). People buy fewer cones on wet days.
- **weekend**: 1 for Saturday or Sunday, 0 for a weekday. Weekends bring more foot traffic.

**Important:** like version 1, the training data below is *made up* to show the steps. The effect sizes (how many cones rain or weekends change) are my assumptions, not findings from a real shop. Replace `df` with real sales data to use this for a real business.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pickle

## 1. Make the training data (replace this with your real data)

In [ ]:
rng = np.random.default_rng(42)
n = 400

df = pd.DataFrame({
    "temperature": rng.uniform(50, 100, n),                  # daily high, F
    "rain_chance": rng.beta(0.8, 2.0, n) * 100,              # percent, mostly low, sometimes high
    "weekend": rng.integers(0, 2, n),                        # 0 = weekday, 1 = weekend
})

# Assumed "true" effects: warmer = more, rain = fewer, weekend = more, plus noise
df["sales"] = (-95 + 4.96 * df["temperature"]
               - 0.6 * df["rain_chance"]
               + 35 * df["weekend"]
               + rng.normal(0, 12, n)).round()
df.head()

## 2. Train the model
The column names and their order here must match the app exactly.

In [ ]:
FEATURES = ["temperature", "rain_chance", "weekend"]
X = df[FEATURES]
y = df["sales"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_train, y_train)

print("R2 on held-out days:", round(model.score(X_test, y_test), 3))
for name, coef in zip(FEATURES, model.coef_):
    print(f"{name:12s} {coef:+.2f} cones per unit")
print("intercept   ", round(model.intercept_, 1))

## 3. Retrain on all the data and save it
The app loads `model.pkl`.

In [ ]:
final_model = LinearRegression().fit(X, y)
with open("model.pkl", "wb") as f:
    pickle.dump(final_model, f)

# Quick check: a warm, dry Saturday vs a warm, rainy Tuesday
test = pd.DataFrame({"temperature": [85, 85], "rain_chance": [5, 80], "weekend": [1, 0]})
print(final_model.predict(test).round())